In [1]:
import numpy as np
import time
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import brentq

In [2]:
# inputs
PSI = 6894.757
PATM = 14.7 * PSI

def make_params():
    R = 296.8; G = 1.4
    return dict(
        R=R, G=G, CV=R/(G - 1), RC=(2/(G + 1))**(G/(G - 1)),
        SCFM=2118.88/(101325/(R*288.7)),
        V_P=9.0*1e-3, P_P0=3900.0*PSI + PATM, T_P0=300.0,
        P_SET=360.0*PSI + PATM,
        CDA_REG=1.0/38.0*6.4516e-4,
        PIN_TAB=np.array([1000.0, 3000.0])*PSI + PATM,
        K_TAB=np.array([3.7, 2.0])*PSI,
        CDA_P=2.05e-5, RHO_F=851.0, V_F=17.1*1e-3, V_U0=1.8*1e-3,
        P_U0=360.0*PSI + PATM, T_U0=300.0,
        CDA_F=1/np.sqrt(1/8.917e-5**2 + 1/(2*2.700e-5)**2), N_BR=2,
        P_C=300.0*PSI + PATM,
    )

def initial_state(p):
    m_p0 = p["P_P0"]*p["V_P"]/(p["R"]*p["T_P0"])
    m_u0 = p["P_U0"]*p["V_U0"]/(p["R"]*p["T_U0"])
    m_f0 = (p["V_F"] - p["V_U0"])*p["RHO_F"]
    return np.array([m_p0, p["T_P0"], m_u0, p["T_U0"], m_f0])

def mdot_orifice(P0, T0, P, CdA, p):
    G, R = p["G"], p["R"]
    r = max(P/P0, p["RC"])
    return CdA*P0*np.sqrt(2*G/((G - 1)*R*T0)*(r**(2/G) - r**((G + 1)/G)))

def n2_flow(P_p, T_p, P_u, p):
    R = p["R"]
    P_top = min(p["P_SET"], P_p)
    if P_u >= P_top:
        return 0.0, P_top
    K = np.interp(P_p, p["PIN_TAB"], p["K_TAB"])*p["SCFM"]
    a = R*T_p/(2*P_u*p["CDA_P"]**2)
    mp = (-K + np.sqrt(K*K + 4*a*(p["P_SET"] - P_u)))/(2*a)
    P_r = p["P_SET"] - K*mp
    if P_r <= P_p and mdot_orifice(P_p, T_p, P_r, p["CDA_REG"], p) >= mp:
        return mp, P_r
    line = lambda Pr: p["CDA_P"]*np.sqrt(2*P_u*max(Pr - P_u, 0.0)/(R*T_p))
    P_r = brentq(lambda Pr: mdot_orifice(P_p, T_p, Pr, p["CDA_REG"], p) - line(Pr), P_u, P_p)
    return line(P_r), P_r

def outputs(y, p):
    m_p, T_p, m_u, T_u, m_f = y
    R = p["R"]
    P_p = m_p*R*T_p/p["V_P"]
    V_u = p["V_F"] - m_f/p["RHO_F"]
    P_u = m_u*R*T_u/V_u
    mp, P_r = n2_flow(P_p, T_p, P_u, p)
    mf = p["CDA_F"]*np.sqrt(2*p["RHO_F"]*max(P_u - p["P_C"], 0.0))
    return P_p, P_r, P_u, mp, mf

def ydot(t, y, p):
    m_p, T_p, m_u, T_u, m_f = y
    P_p, P_r, P_u, mp, mf = outputs(y, p)
    G = p["G"]
    return np.array([
        -mp,
        -(G - 1)*T_p*mp/m_p,
        mp,
        (mp*(G*T_p - T_u) - P_u*(mf/p["RHO_F"])/p["CV"])/m_u,
        -mf,
    ])

In [10]:
# --- Implementation Comparison ---
def compare_implementations():
    p = make_params()
    y0 = initial_state(p)
    t_span = (0.0, 20.0)
    dt_rk4 = 0.01

    print(f"--- Performance Comparison (Simulation up to t={t_span[1]}s) ---")

    # 1. Exact Reference (Radau - Implicit, high tolerance)
    sol_exact = solve_ivp(lambda t, y: ydot(t, y, p), t_span, y0, method='Radau', rtol=1e-11, atol=1e-11, t_eval=[t_span[1]])
    y_exact = sol_exact.y[:, -1]

    # 2. Custom RK4 (fixed step, explicit, order 4; same as integrators.step_rk4)
    def rk4_step(t, y, h):
        k1 = ydot(t, y, p)
        k2 = ydot(t + 0.5*h, y + 0.5*h*k1, p)
        k3 = ydot(t + 0.5*h, y + 0.5*h*k2, p)
        k4 = ydot(t + h, y + h*k3, p)
        return y + h*(k1 + 2*k2 + 2*k3 + k4)/6

    t0_rk4 = time.perf_counter()
    y_rk4 = y0.copy()
    t, nfev_rk4 = t_span[0], 0
    while t < t_span[1] - 1e-9:
        h = min(dt_rk4, t_span[1] - t)          # clip last step to land on t_end
        y_rk4 = rk4_step(t, y_rk4, h)
        t += h
        nfev_rk4 += 4                           # 4 RHS evaluations per step
    t_rk4_elapsed = time.perf_counter() - t0_rk4
    err_rk4 = np.linalg.norm(y_rk4 - y_exact)

    print(f"\n1. Custom RK4 (dt={dt_rk4}):")
    print(f"   Execution Time: {t_rk4_elapsed:.4f} seconds")
    print(f"   Error vs Exact: {err_rk4:.5e}")
    print(f"   Function Evaluations: {nfev_rk4}")

    # 3. Standard Library Implementation: SciPy RK45 (Explicit Runge-Kutta)
    t0_rk45 = time.perf_counter()
    sol_rk45 = solve_ivp(lambda t, y: ydot(t, y, p), t_span, y0, method='RK45')
    t_rk45_elapsed = time.perf_counter() - t0_rk45
    err_rk45 = np.linalg.norm(sol_rk45.y[:, -1] - y_exact)

    print(f"\n2. Standard Library (SciPy RK45):")
    print(f"   Execution Time: {t_rk45_elapsed:.4f} seconds")
    print(f"   Error vs Exact: {err_rk45:.5e}")
    print(f"   Function Evaluations: {sol_rk45.nfev}")

    # 4. Standard Library Implementation: SciPy LSODA (Adams/BDF Switching)
    t0_lsoda = time.perf_counter()
    sol_lsoda = solve_ivp(lambda t, y: ydot(t, y, p), t_span, y0, method='LSODA')
    t_lsoda_elapsed = time.perf_counter() - t0_lsoda
    err_lsoda = np.linalg.norm(sol_lsoda.y[:, -1] - y_exact)

    print(f"\n3. Standard Library (SciPy LSODA):")
    print(f"   Execution Time: {t_lsoda_elapsed:.4f} seconds")
    print(f"   Error vs Exact: {err_lsoda:.5e}")
    print(f"   Function Evaluations: {sol_lsoda.nfev}")

if __name__ == "__main__":
    compare_implementations()

--- Performance Comparison (Simulation up to t=20.0s) ---

1. Custom RK4 (dt=0.01):
   Execution Time: 0.1684 seconds
   Error vs Exact: 5.83171e-09
   Function Evaluations: 8000

2. Standard Library (SciPy RK45):
   Execution Time: 0.0124 seconds
   Error vs Exact: 7.94326e-01
   Function Evaluations: 164

3. Standard Library (SciPy LSODA):
   Execution Time: 0.0028 seconds
   Error vs Exact: 1.08304e-01
   Function Evaluations: 92
